In [1]:
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
import os

from specula.mmlib.yaml_overrides import write_yaml_overrides
from specula.mmlib.compute_rec import compute_and_save_rec
from specula.mmlib.utils import get_pupil_mask

2026-11-04 09:47:50,875 [INFO]: [specula]: Cupy import successfull. Installed version is: 13.3.0
2026-11-04 09:47:51,079 [INFO]: [specula]: Default device is GPU number 0


In [2]:
import matplotlib as mpl
from cycler import cycler

cmap = mpl.colormaps['hot'] # 1. Choose your colormap (e.g., 'hot', 'jet', 'turbo')
colors = cmap(np.linspace(0.0, 1.0, 7)) # 2. Extract a set of colors from it (e.g., 6 distinct colors across the map)
# plt.rcParams['axes.prop_cycle'] = cycler(color=colors) # 3. Apply it to the default line cycle
plt.rcParams['image.cmap'] = 'magma' # 4. Change the default colormap for images/scatter plots
plt.rcParams.update({
    "text.usetex": True, # Use LaTeX to write all text
    "font.family": "serif", # Use a serif font
    "font.serif": ["Computer Modern Roman"] # Specifically use the classic LaTeX font
})
plt.rcParams['image.origin'] = 'lower'

In [3]:
nSubaps = [10,14,24,36,48]
rMods = [0,0.5,1,2]
# nModes = [54,100,300,800,1200,1600]
nModes = [54,100,300,800,1300]
wvls = [900,1300]
overrides_yaml = 'temp_overrides'

calib_dir = '/raid1/mmenessini/calibration/WFSW'
result_dir = '/raid1/mmenessini/results/WFSW'

In [4]:
# Generate pupil data
for nSubap in nSubaps:
    tag = f'pyr_pupdata_{nSubap}x{nSubap}'
    try:
        pupdata = fits.getdata(os.path.join(calib_dir,'pupils',tag+'.fits'))
    except FileNotFoundError:
        ovdes = ("{"
            f"pyr_vis.pup_diam: {nSubap}, "
            f"pyr_vis.pup_dist: {np.max((14,np.ceil(nSubap*60/48)))}, "
            f"pyr_pupdata.output_tag: {tag}, "
            "}")
        write_yaml_overrides(ovdes,temp_name=overrides_yaml)
        os.system(f'specula ristretto_main.yml calib_pupdata.yml {overrides_yaml}.yml')

In [5]:
# Generate IM data
for wvl in wvls:
    for rMod in rMods:
        for nSubap in nSubaps:
            tag = f'pyr{rMod:1.1f}_{nSubap:1.0f}x{nSubap:1.0f}_wvl{wvl:1.0f}_bmc2k_vlt'
            im_tag = tag+'_im'
            try:
                im = fits.getdata(os.path.join(calib_dir,'im',im_tag+'.fits'),1)
            except FileNotFoundError:
                print(f'Simulating: {nSubap}x{nSubap}, {wvl}nm, {rMod:1.1f} '+'$\lambda$/D')
                ovdes = ("{"
                    f"pyr_vis.pup_diam: {nSubap}, "
                    f"pyr_vis.pup_dist: {np.max((14,np.ceil(nSubap*60/48)))}, "
                    f"source_vis.wavelengthInNm: {wvl}, "
                    f"pyr_vis.mod_amp: {rMod:1.1f}, "
                    f"pyr_im_calibrator.im_tag: {im_tag}, "
                    f"pyr_vis.wavelengthInNm: {wvl}, "
                    f"pyr_slopes_vis.pupdata_object: pyr_pupdata_{nSubap}x{nSubap}, "
                    "}")
                write_yaml_overrides(ovdes,temp_name=overrides_yaml)
                os.system(f'specula ristretto_main.yml calib_im.yml {overrides_yaml}.yml')

                for N in nModes:
                    rec_tag = tag+f'_{N:1.0f}modes_rec'
                    compute_and_save_rec(root_dir=calib_dir, im_tag=im_tag, rec_tag=rec_tag, Nmodes=N)

In [6]:
mask = get_pupil_mask(npix=120,filepath=os.path.join(calib_dir,'pupils','pyr_pupdata_48x48.fits'))
os.makedirs(os.path.join(calib_dir,'aliasing'),exist_ok=True)

for rMod in rMods:
    for j,nSubap in enumerate(nSubaps):
        for i,N in enumerate(nModes):
            if i <= j:
                tag = f'pyr{rMod:1.1f}_{nSubap:1.0f}x{nSubap:1.0f}'
                alias_vis_tag = tag+f'_{N}modes_wvl900_alias'
                frame_vis_tag = tag+f'_{N}modes_wvl900_frame'
                alias_nir_tag = tag+f'_{N}modes_wvl1300_alias'
                frame_nir_tag = tag+f'_{N}modes_wvl1300_frame'
                try:
                    alias_vis = fits.getdata(os.path.join(calib_dir,'aliasing',alias_vis_tag+'.fits'))
                    frames_vis = fits.getdata(os.path.join(calib_dir,'aliasing',frame_vis_tag+'.fits'))
                    alias_nir = fits.getdata(os.path.join(calib_dir,'aliasing',alias_nir_tag+'.fits'))
                    frames_nir = fits.getdata(os.path.join(calib_dir,'aliasing',frame_nir_tag+'.fits'))
                except FileNotFoundError:
                    print(f'Simulating: {nSubap}x{nSubap}, {wvl}nm, {rMod:1.1f} '+'$\lambda$/D')
                    ovdes = ("{"
                        f"pyr_vis.pup_diam: {nSubap}, "
                        f"pyr_vis.pup_dist: {np.max((14,np.ceil(nSubap*60/48)))}, "
                        f"pyr_vis.mod_amp: {rMod:1.1f}, "
                        f"pyr_slopes_vis.pupdata_object: pyr_pupdata_{nSubap}x{nSubap}, "
                        f"pyr_modes_vis.recmat_object: {tag}_wvl900_bmc2k_vlt_{N}modes_rec, "
                        f"pyr_nir.pup_diam: {nSubap}, "
                        f"pyr_nir.pup_dist: {np.max((14,np.ceil(nSubap*60/48)))}, "
                        f"pyr_nir.mod_amp: {rMod:1.1f}, "
                        f"pyr_slopes_nir.pupdata_object: pyr_pupdata_{nSubap}x{nSubap}, "
                        f"pyr_modes_nir.recmat_object: {tag}_wvl1300_bmc2k_vlt_{N}modes_rec, "
                        "}")
                    write_yaml_overrides(ovdes,temp_name=overrides_yaml)
                    os.system(f'specula ristretto_main.yml calib_wfs_aliasing.yml {overrides_yaml}.yml')

                    alias_vis = fits.getdata(os.path.join(result_dir,'meas_vis.fits'))
                    frames_vis = fits.getdata(os.path.join(result_dir,'vis_frames.fits'))
                    alias_nir = fits.getdata(os.path.join(result_dir,'meas_nir.fits'))
                    frames_nir = fits.getdata(os.path.join(result_dir,'nir_frames.fits'))

                    fits.writeto(os.path.join(calib_dir,'aliasing',alias_vis_tag+'.fits'),alias_vis)
                    fits.writeto(os.path.join(calib_dir,'aliasing',alias_nir_tag+'.fits'),alias_nir)
                    fits.writeto(os.path.join(calib_dir,'aliasing',frame_vis_tag+'.fits'),frames_vis)
                    fits.writeto(os.path.join(calib_dir,'aliasing',frame_nir_tag+'.fits'),frames_nir)
